# Comparaison des modèles et décisions de modélisation

## Une métrique événementielle

Le système évalue chaque machine toutes les 30 minutes. Compter les fausses alertes par pas pénaliserait plusieurs
fois le même dérangement : une machine en alerte 10 h d'affilée, c'est une seule visite du technicien. On suit donc :

- le **recall événementiel** : part des pannes précédées d'au moins une alerte dans les 120 h
- le **préavis** : temps entre le début de la première alerte de la fenêtre et la panne, borné à 120 h
- les **fausses alertes par machine et par jour**, comptées en épisodes : une suite d'alertes = une intervention,
  redécoupée toutes les 24 h pour qu'alerter en permanence ne paie pas

Toutes les alertes tombées dans les 120 h avant une panne lui reviennent. Avant, seule la dernière tranche de 24 h
était rattachée : le préavis ne pouvait pas dépasser 24 h et une alerte précoce, pourtant juste, comptait comme
fausse, ce qui poussait le seuil à n'alerter qu'au dernier moment.

## Protocole

- 03 fenêtres de validation (T2, T3 et T4 2024), chacune entraînée sur ce qui précède, avec un tampon de 120 h
  pour que la cible ne déborde pas sur la période suivante
- Test sur 2025, avec le seuil choisi sur la dernière fenêtre, appliqué tel quel
- 1 machine sur cinq n'est jamais vue à l'entraînement : elle mesure la généralisation à une machine neuve

In [1]:
import sys
import time

sys.path.append("..")

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.compose import make_column_transformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

from src.config import CIBLE
from src.donnees import FEATURES, MODES_PANNE_PREDICTIBLES, PLIS, charger, decouper, decouper_pli, preparer_X
from src.evaluation import (
    BUDGET_FAUSSES_ALERTES_PAR_MACHINE_JOUR,
    PAS_HORIZON,
    bilan_economique,
    grille_seuils,
    metriques_evenementielles,
    trouver_seuil_pour_budget_fa,
)
from src.train import HYPERPARAMETRES_LIGHTGBM, entrainer, nouveau_modele

df = charger()
_, _, test = decouper(df)

## Les candidats, du plus simple au plus complexe

In [2]:
numeriques = [f for f in FEATURES if f != "type_machine"]
encodeur = lambda numerique: make_column_transformer(
    (numerique, numeriques), (OneHotEncoder(handle_unknown="ignore"), ["type_machine"])
)
CANDIDATS = {
    "Régression logistique": lambda: make_pipeline(
        encodeur(make_pipeline(SimpleImputer(strategy="median"), StandardScaler())),
        LogisticRegression(max_iter=3000, class_weight="balanced"),
    ),
    "Arbre de décision": lambda: make_pipeline(
        encodeur("passthrough"), DecisionTreeClassifier(max_depth=6, min_samples_leaf=1000, random_state=42)
    ),
    "HistGradientBoosting": lambda: HistGradientBoostingClassifier(
        max_iter=80, learning_rate=0.1, max_leaf_nodes=7, min_samples_leaf=1000, l2_regularization=5.0,
        categorical_features="from_dtype", random_state=42,
    ),
    "LightGBM (déployé)": nouveau_modele,
}

In [3]:
def recall_au_budget(fenetre, scores):
    return trouver_seuil_pour_budget_fa(
        fenetre, scores, grille_seuils(scores), PAS_HORIZON, BUDGET_FAUSSES_ALERTES_PAR_MACHINE_JOUR
    )


def evaluer(fabrique):
    ligne, pannes_validation = {}, 0
    for i, (debut, fin) in enumerate(PLIS, start=1):
        train, validation = decouper_pli(df, debut, fin)
        depart = time.perf_counter()
        train = train[train["etat"] == "marche"]  # le modèle n'apprend que sur les machines en marche
        modele = fabrique().fit(preparer_X(train), train[CIBLE])
        ligne["entrainement_s"] = time.perf_counter() - depart
        seuil, m = recall_au_budget(validation, modele.predict_proba(preparer_X(validation))[:, 1])
        ligne[f"recall_T{i + 1}_2024"] = m["recall_evenementiel"]
        pannes_validation += m["nb_pannes"]
    # Protocole de déploiement : le seuil de la dernière fenêtre, appliqué tel quel au test 2025
    # (l'évaluation n'alerte jamais une machine arrêtée)
    m = metriques_evenementielles(test, modele.predict_proba(preparer_X(test))[:, 1], seuil, PAS_HORIZON)
    recall_validation = np.mean([ligne[f"recall_T{i + 1}_2024"] for i in range(1, len(PLIS) + 1)])
    return ligne | {
        "recall_validation": recall_validation,
        # Écart-type binomial : sous ~2 écarts-types, deux modèles ne sont pas départagés
        "ecart_type_validation": np.sqrt(recall_validation * (1 - recall_validation) / pannes_validation),
        "recall_test_2025": m["recall_evenementiel"],
        "ecart_type_test": np.sqrt(m["recall_evenementiel"] * (1 - m["recall_evenementiel"]) / m["nb_pannes"]),
        "recall_preavis_4h": m["recall_preavis_4h"],
        "precision_alertes": m["precision_alertes"],
        "fa_test": m["taux_fausses_alertes_par_machine_jour"],
        "gain_net_M_fcfa": bilan_economique(test, m)["gain_net"] / 1e6,
    }


resultats = pd.DataFrame({nom: evaluer(fabrique) for nom, fabrique in CANDIDATS.items()}).T
colonnes = ["recall_validation", "ecart_type_validation", "recall_test_2025", "ecart_type_test", "recall_preavis_4h"]
colonnes += ["precision_alertes", "fa_test", "gain_net_M_fcfa", "entrainement_s"]
resultats[colonnes].astype(float).round(3)

,recall_validation,ecart_type_validation,recall_test_2025,ecart_type_test,recall_preavis_4h,precision_alertes,fa_test,gain_net_M_fcfa,entrainement_s
Régression logistique,0.778,0.031,0.788,0.024,0.784,0.239,0.174,5.761,2.312
Arbre de décision,0.824,0.029,0.795,0.024,0.792,0.207,0.185,5.450,6.847
HistGradientBoosting,0.795,0.030,0.799,0.024,0.799,0.241,0.168,7.342,2.383
LightGBM (déployé),0.818,0.029,0.806,0.024,0.806,0.226,0.182,8.070,1.315


## Sur-apprentissage

Les relevés d'une même machine se ressemblent d'un pas de 30 min à l'autre : un modèle trop libre apprend les
séquences du train par cœur. On compare l'ancien réglage (150 arbres de 15 feuilles, 80 relevés par feuille) au
réglage déployé, puis on suit la perte en validation selon le nombre d'arbres.

In [4]:
train, validation = decouper_pli(df, *PLIS[-1])
train, validation = train[train["etat"] == "marche"], validation[validation["etat"] == "marche"]
ANCIEN = HYPERPARAMETRES_LIGHTGBM | {"n_estimators": 150, "num_leaves": 15, "min_child_samples": 80, "reg_lambda": 0.3}
lignes = {}
for nom, params in {"Ancien réglage": ANCIEN, "Réglage déployé": HYPERPARAMETRES_LIGHTGBM}.items():
    modele = LGBMClassifier(**params, random_state=42, n_jobs=-1, verbose=-1).fit(preparer_X(train), train[CIBLE])
    lignes[nom] = {
        f"pr_auc_{nom_jeu}": average_precision_score(jeu[CIBLE], modele.predict_proba(preparer_X(jeu))[:, 1])
        for nom_jeu, jeu in (("train", train), ("validation", validation))
    }
pd.DataFrame(lignes).T.round(3)

,pr_auc_train,pr_auc_validation
Ancien réglage,0.893,0.353
Réglage déployé,0.599,0.398


In [5]:
modele = LGBMClassifier(**(HYPERPARAMETRES_LIGHTGBM | {"n_estimators": 400}), random_state=42, n_jobs=-1, verbose=-1)
modele.fit(preparer_X(train), train[CIBLE], eval_set=[(preparer_X(train), train[CIBLE]), (preparer_X(validation), validation[CIBLE])])
pertes = pd.DataFrame({"train": modele.evals_result_["valid_0"]["binary_logloss"],
                       "validation": modele.evals_result_["valid_1"]["binary_logloss"]}, index=range(1, 401))  # fmt: skip
pertes.loc[[25, 50, 80, 150, 250, 400]].round(4)

,train,validation
25,0.1834,0.1946
50,0.1726,0.1931
80,0.1626,0.1953
150,0.1461,0.2001
250,0.1288,0.2047
400,0.1101,0.2115


## Ce qui compte pour le modèle

In [6]:
train, _, _ = decouper(df)
train = train[train["etat"] == "marche"]
lgbm = nouveau_modele().fit(preparer_X(train), train[CIBLE])
gain = pd.Series(lgbm.booster_.feature_importance("gain"), index=FEATURES)
(gain / gain.sum()).sort_values(ascending=False).head(12).round(3)

vibration_moyenne_24h                  0.492
vibration_max_24h                      0.139
heures_depuis_changement_outil         0.116
heures_depuis_maintenance              0.076
type_machine                           0.060
vibration                              0.040
temperature_process_max_24h            0.019
concentration_poussiere_moyenne_24h    0.016
concentration_poussiere_max_24h        0.007
couple_max_24h                         0.007
ecart_thermique_max_24h                0.005
couple_moyenne_24h                     0.004
dtype: float64

## Recall par mode de panne sur le test 2025 (modèle déployé)

In [7]:
calibre, seuil, _ = entrainer(df)
probas = calibre.predict_proba(preparer_X(test))[:, 1]
pd.DataFrame({
    mode: {k: m[k] for k in ("nb_pannes", "nb_attrapees", "recall_evenementiel")}
    | {"preavis_median_h": float(np.median(m["preavis_heures"]))}
    for mode in MODES_PANNE_PREDICTIBLES
    for m in [metriques_evenementielles(test, probas, seuil, PAS_HORIZON, modes_panne=(mode,))]
}).T.round(3)

,nb_pannes,nb_attrapees,recall_evenementiel,preavis_median_h
TWF,108.0,108.0,1.000,120.0
HDF,77.0,55.0,0.714,90.0
OSF,98.0,65.0,0.663,108.0


## Décisions de modélisation

- **LightGBM conservé** : les quatre modèles annoncent entre 78 et 82 % des pannes en validation, à moins de
  1,5 écart-type les uns des autres. LightGBM est le plus rapide à entraîner et SHAP l'explique exactement.
- **Seulement les machines en marche** : à l'arrêt (week-ends), en panne ou en délestage, les capteurs tombent à
  zéro. Le modèle n'apprend que sur les relevés en marche et n'alerte jamais une machine qui ne tourne pas ; la cible,
  elle, est calculée sur toute la chronologie.
- **Statistiques sur les seuls relevés en marche** : le week-end, les capteurs tombent à zéro ; calculées sur
  toute la chronologie, les moyennes sur 24 h du lundi matin mélangeaient mesures et zéros. Calculées sur les seuls
  relevés en marche, les mêmes 35 features font passer le test 2025 de 64 % à 81 % des pannes annoncées (surchauffe
  de 44 % à 71 %, surcharge de 48 % à 66 %), au même seuil de fausses alertes.
- **Features simples, sur 24 h** : des features de dérive lente (moyennes sur 7 jours, résidu thermique, expositions
  cumulées) ont été testées puis écartées. Elles copiaient de trop près les mécanismes du simulateur et imposaient
  7 jours d'historique à l'API ; l'essentiel de leur gain venait du calcul sur les relevés en marche, repris ici.
- **Régularisation forte** : 80 arbres de 7 feuilles, au moins 1 000 relevés par feuille, L2 = 5, toutes les features à chaque arbre. L'ancien réglage
  apprenait le train par cœur (PR-AUC 0,89 sur le train contre 0,35 en validation) et sa perte en validation
  remontait dès 50 arbres. Le réglage retenu a la meilleure PR-AUC en validation (0,40 contre 0,35).
- **Pas de lissage du risque** : une moyenne glissante sur 6 h n'apportait que quelques points de recall, au prix
  d'une API qui doit noter tout l'historique. Une prédiction = un relevé.
- **Budget de fausses alertes et coûts** (`src/metier.py`) : une panne annoncée devient un arrêt planifié de 2 h
  sans rebut (environ 75 000 FCFA évités en moyenne sur les pannes prévisibles), une alerte coûte une heure de
  technicien (3 250 FCFA). Sur 5 jours d'horizon, une panne possible peut coûter jusqu'à 5 inspections : le modèle
  est rentable dès qu'environ une alerte sur cinq est utile. Le budget de 0,2 fausse alerte par machine et par jour
  reste le critère du seuil, le gain net sert à la promotion.
- **Promotion décidée sur la validation** : le @champion n'est remplacé que si le gain net en validation ne baisse
  pas ; sinon il reste en service et une promotion manuelle passe par une revue. Le test 2025 ne sert qu'au rapport.
- **Pas de standardisation, catégorie native** : un modèle à base d'arbres est insensible aux transformations
  monotones ; LightGBM gère les valeurs manquantes et la gamme (`type_machine`) nativement. Les catégories sont
  fixées (L, M, H) pour qu'une machine prédite seule soit lue comme à l'entraînement.
- **Calibration sigmoïde sur les log-odds** : elle rend les probabilités interprétables (sur 100 machines à 10 %,
  une dizaine tombe en panne dans les 5 jours), sans le plancher qu'imposait une calibration sur la probabilité brute.